<a href="https://colab.research.google.com/github/robertbarcik/ADK-tutorial/blob/main/notebooks/01_mental_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 01 — From function calling to agents

> **⚡ Quick path** — this is one of four modules on the 1-hour course preview.
> See the [README's Quick path section](../README.md#-quick-path---1-hour) for the sequence.

> **Where you are** — course 2 of the path, right after *Úvod do GenAI v Pythone*.
> - **You can already:** call a model with `client.responses.create(...)`, hand-write a tool
>   schema, and run your own function-calling loop (`chat_with_function_execution`,
>   `weather_assistant` — notebook 5 of that course).
> - **New in this module:** the four ADK primitives (`Agent`, `Runner`, `Event`, `Session`)
>   and your first taste of `async`.
> - **First met here, used everywhere later:** `await`, the event stream.

At the end of the previous course, in the function-calling chapter, we made a promise: function
calling is probably the most useful capability of large language models — and there is a ladder
above it. Function calling got popularized by a wave of open-source agents; **MCP** standardized
how tools are offered to models; and Google's **Agent Development Kit (ADK)** moved one layer up,
into *orchestration* — where even calling another agent is just a function call. This course is
that promised next rung.

Here is the claim this module will prove: **you already built an agent in the previous course
— you just didn't call it that.** `weather_assistant()` was a loop that let a model decide to
call your Python function and answer with the result. In this notebook we first rebuild that
loop in two cells (your code, one changed line), then build the same behavior in ADK and map
your loop's parts onto ADK's four primitives, piece by piece.

**What you'll leave with:** a mental model for the four primitives ADK gives you (`Agent`,
`Runner`, `Event`, `Session`) — and the knowledge that none of them is magic, because you have
already written the naive version of each by hand.

**Runs in:** Google Colab or a local Python 3.10+ environment.
**Running cost:** well under $0.01.


# Setup

Install dependencies, configure authentication, and import ADK.

## Install Dependencies

Run the cell below to install everything this module needs.

In [1]:
!pip install -q google-adk==2.7.1 openai==2.54.0 litellm==1.85.7 python-dotenv==1.0.1 nest-asyncio==1.6.0 deprecated==1.2.18 2>/dev/null

print("✅ Packages installed.")

✅ Packages installed.


## API Key Configuration — why not your `OPENAI_API_KEY`?

Fair question. In the previous course everything ran on one `OPENAI_API_KEY`. This course asks
you for one new key — **OpenRouter** ([openrouter.ai/keys](https://openrouter.ai/keys)) — and
here is the honest reason:

Your OpenAI key opens exactly one vendor. This course's whole point is that ADK is
**vendor-neutral**: in Module 04 the *same agent* will run on OpenAI, Anthropic, Google, Qwen and
Meta models — and OpenRouter is one key, one small balance, every vendor. It resells all of them
at (nearly) the providers' own prices; you heard it mentioned in the previous course's video on
local models and platforms. The entire Part 1 of this course costs a couple of dollars.

The setup ritual is exactly the one you know:

**Method 1 (Colab, recommended):** Click the 🔑 icon in the left sidebar → Add new secret →
Name: `OPENROUTER_API_KEY` → Value: your key → enable notebook access. (Same steps as your
`OPENAI_API_KEY` — it can stay there; they don't conflict.)

**Method 2 (local or fallback):** Put it in a `.env` file next to the notebook (new in this
course — a plain text file with `OPENROUTER_API_KEY=sk-or-...` per line; the code below reads
it), or just paste it when prompted.


In [2]:
import os

# Try Colab secrets first, then .env, then prompt.
OPENROUTER_API_KEY = None

try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
    print("✅ API key loaded from Colab secrets.")
except Exception:
    try:
        from dotenv import load_dotenv
        load_dotenv()
        OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
        if OPENROUTER_API_KEY:
            print("✅ API key loaded from .env file.")
    except ImportError:
        pass

if not OPENROUTER_API_KEY:
    from getpass import getpass
    print("💡 Tip for Colab users: Go to 🔑 (left sidebar) → Add new secret → Name: OPENROUTER_API_KEY")
    OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key: ")

assert OPENROUTER_API_KEY and OPENROUTER_API_KEY.strip(), "❌ No API key provided."
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

# Cheap + fast model for all vendor-agnostic demos. Swap it to try another provider.
MODEL_STRING = "openrouter/openai/gpt-5.6-luna"
print(f"✅ Model: {MODEL_STRING}")

✅ API key loaded from .env file.
✅ Model: openrouter/openai/gpt-5.6-luna


# The Agent You Already Built

Notebook 5 of the previous course. You wrote:

- `get_current_weather(location: str, format: str) -> dict` — a plain Python function,
- a hand-written `tools` list — the JSON schema telling the model when and how to call it,
- `available_functions = {"get_current_weather": get_current_weather}` — your dispatch dictionary,
- `chat_with_function_execution(messages, tools, available_functions)` — the five-step loop:
  call the model → find `function_call` items → `json.loads` the arguments → execute via the
  dictionary → send the result back for a final answer,
- and `weather_assistant(user_query)` wrapping it all.

Let's rebuild its essence in two cells. **The only surprise is one line:** `base_url`. OpenRouter
speaks the same Responses API your code already speaks, so the `OpenAI` client class works
unchanged — you point it at a different address and the model string gains a vendor prefix
(`openai/gpt-5.6-luna` — "OpenAI's gpt-5.6-luna, please"). That's what "one key, every vendor"
means in practice.


In [3]:
from openai import OpenAI
import json

# The one changed line vs the previous course: base_url points at OpenRouter.
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_API_KEY)
OPENAI_MODEL = "openai/gpt-5.6-luna"   # vendor prefix: OpenAI's model, served via OpenRouter

# Your function from notebook 5 — same signature, same docstring style.
# One change: deterministic data instead of random, so re-runs are comparable.
FAKE_WEATHER = {
    "Bratislava": ("Sunny", 21.0),
    "Prague":     ("Cloudy", 14.0),
    "Munich":     ("Rainy", 11.0),
}

def get_current_weather(location: str, format: str) -> dict:
    """Get the current weather for a location.

    Args:
        location (str): The city, e.g. "Bratislava" or "Prague".
        format (str): The temperature format, either 'celsius' or 'fahrenheit'.

    Returns:
        dict: location, temperature, unit and condition (mock data).
    """
    city = location.split(",")[0].strip()
    condition, celsius = FAKE_WEATHER.get(city, ("Unknown", None))
    if celsius is None:
        return {"location": location, "condition": "Unknown", "error": f"No data for {location}."}
    temp = celsius if format == "celsius" else round(celsius * 9 / 5 + 32, 1)
    return {"location": location, "temperature": temp, "unit": format, "condition": condition}

# Your hand-written schema from notebook 5 — every field written by you, by hand.
tools = [{
    "type": "function",
    "name": "get_current_weather",
    "description": "Get the current weather in a given location. Use this when the user asks "
                   "about current or present weather conditions.",
    "parameters": {
        "type": "object",
        "properties": {
            "location": {"type": "string",
                         "description": "The city, e.g. 'Bratislava' or 'Prague'."},
            "format": {"type": "string", "enum": ["celsius", "fahrenheit"],
                       "description": "The temperature unit to use."},
        },
        "required": ["location", "format"],
    },
}]

# Your dispatch dictionary: schema name -> actual Python function.
available_functions = {"get_current_weather": get_current_weather}

print("✅ Your notebook-5 setup, rebuilt. One changed line: base_url.")


✅ Your notebook-5 setup, rebuilt. One changed line: base_url.


In [4]:
def chat_with_function_execution(messages, tools, available_functions):
    """Your five-step loop from notebook 5, condensed. Same steps, same order."""
    # Step 1: initial API call — the model sees the schemas and decides.
    response = client.responses.create(model=OPENAI_MODEL, input=messages,
                                       tools=tools, tool_choice="auto")
    function_calls = [item for item in response.output
                      if getattr(item, "type", None) == "function_call"]
    if not function_calls:
        return response.output_text  # no tool needed — plain answer

    for tool_call in function_calls:
        # Step 2: extract what the model wants ('arguments' is a JSON *string* — parse it).
        function_arguments = json.loads(tool_call.arguments)
        print(f"[model asked for] {tool_call.name}({function_arguments})")
        # Step 3: execute the real Python function via your dispatch dict.
        function_result = available_functions[tool_call.name](**function_arguments)
        print(f"[function returned] {function_result}")
        # Step 4: send the result back. (Notebook 5's shortcut, kept honestly: we smuggle
        # the result in as a fake *user* message instead of a proper tool-result message.)
        messages.append({"role": "user",
                         "content": f"Function {tool_call.name} returned: {json.dumps(function_result)}"})

    # Step 5: final call — the model writes the answer using the result.
    final_response = client.responses.create(model=OPENAI_MODEL, input=messages)
    return final_response.output_text

messages = [
    {"role": "system", "content": "You are a helpful weather assistant. Use the provided "
                                  "functions to get accurate weather data."},
    {"role": "user", "content": "What's the weather in Bratislava right now? Use Celsius."},
]
answer = chat_with_function_execution(messages, tools, available_functions)
print(f"\n🤖 {answer}")


[model asked for] get_current_weather({'location': 'Bratislava', 'format': 'celsius'})
[function returned] {'location': 'Bratislava', 'temperature': 21.0, 'unit': 'celsius', 'condition': 'Sunny'}



🤖 Bratislava is currently **21°C and sunny**.


## This Worked — and It Is the Ceiling of That Approach

Look at what you just ran with a builder's eye:

- **One round only.** If the model wanted a second tool after seeing the first result, your loop
  wouldn't notice — it goes straight to the final answer.
- **The result travels as a fake `user` message.** The Responses API has a proper tool-result
  message type; notebook 5 skipped it to stay simple. Fine for learning, wrong for production.
- **You re-write this loop for every project.** And everyone's hand-rolled loop is slightly
  different, so nothing is reusable — no shared logging, no shared tests.
- **No conversation memory.** `messages` lives in one cell. Two users, or one user coming back
  tomorrow? Your problem.
- **Nothing standard to observe.** Your `print()`s are the only window into what happened.

ADK is what you get when this loop is written once, properly, by a team that hit all five
problems. Every piece of your loop maps onto an ADK primitive:

| Your notebook-5 code | ADK | The upgrade |
|---|---|---|
| Hand-written `tools` schema | `tools=[get_current_weather]` | ADK **generates the schema from the docstring + type hints** you already write |
| `available_functions` dict | built-in dispatch | name→function routing, argument parsing, error wrapping |
| `system` message | `instruction=` | plus templating from state (M05) |
| Your Steps 1–5 loop | `Runner` | loops until done, any number of tool rounds |
| `messages` list | `Session` | stored, multi-user, survives restarts (M03, M08) |
| Your `print()`s | `Event` stream | every step is a typed, inspectable object |

Keep this table in mind — the rest of the module walks it row by row.


## Import Libraries — and what LiteLLM actually is

One thing to clear up before the imports, because it confuses almost everyone on day one.

ADK is Google's framework. Out of the box it knows how to talk to exactly one family of models: Gemini. If you write `LlmAgent(model="gemini-2.5-flash")`, ADK calls Google's API itself — nothing extra to install, nothing to wrap.

We are not doing that. We want the *same code* to run on GPT, Claude, Gemini, Qwen — whatever is cheapest or best next year. The problem: every provider has a slightly different API. Different URL, different JSON shape for the messages, different way of describing tools. Two cells ago you dodged this problem with the `base_url` trick — but that trick only works for providers that imitate OpenAI's API shape. **LiteLLM is that trick, generalized**: a translation layer for 100+ providers, including the ones that don't speak OpenAI.

**LiteLLM** (the library, lowercase `litellm` in the pip cell) is a separate open-source Python package that already did that work. You hand it a model name as one string — `"openrouter/openai/gpt-5.6-luna"` — and it knows which URL to call, how to reshape the request, and how to reshape the response back. One function, 100+ providers. It is *not* part of ADK and it is not made by Google.

**`LiteLlm`** (the class, capital L, from `google.adk.models.lite_llm`) is the bridge. ADK expects a "model object" with a fixed interface — *send these messages and these tool definitions, give me back the model's reply*. `LiteLlm(model="...")` is a small adapter class that implements that interface on the ADK side and calls the LiteLLM library on the other side.

So, to answer the question you probably have: **we are not changing any settings inside ADK, and we are not writing our own model code.** We build one adapter object and plug it into the `model=` slot of the agent. That is the whole trick.

One naming gotcha before you see the string: in the bridge cells you wrote `openai/gpt-5.6-luna` (OpenRouter's own naming). Inside LiteLLM the same model gets one more prefix — `openrouter/openai/gpt-5.6-luna` — because LiteLLM needs to know *which provider* to route through before it can use OpenRouter's model name. The model string has three parts:

```
openrouter / openai/gpt-5.6-luna
   │            │
   │            └── the model, named the way OpenRouter names it (vendor/model)
   └── which LiteLLM "provider" to use = which API endpoint + which API key
```

[OpenRouter](https://openrouter.ai) is a paid reseller: one key, one bill, every vendor's models behind one endpoint. That is why the whole first part of the course needs only one API key.

The full chain, for one user message:

```
your code  →  LlmAgent  →  LiteLlm adapter (ADK)  →  litellm library  →  OpenRouter (HTTPS)  →  the vendor's model
                                                                                        ←  reply comes back the same way
```

The import cell below also silences some noisy-but-harmless warnings that LiteLLM prints at import time. Ignore that part; it is housekeeping, not ADK.


In [5]:
import os
# Silence harmless import-time warnings before heavy imports.
import warnings, io, contextlib
warnings.filterwarnings("ignore")

with contextlib.redirect_stderr(io.StringIO()):
    os.environ.setdefault("LITELLM_LOG", "ERROR")  # silence LiteLLM's import-time provider warnings
    import litellm
    litellm.suppress_debug_info = True
    import logging
    logging.getLogger("LiteLLM").setLevel(logging.WARNING)

    # Core ADK
    from google.adk.agents import LlmAgent
    from google.adk.runners import Runner
    from google.adk.sessions import InMemorySessionService
    from google.adk.models.lite_llm import LiteLlm
    from google.genai import types

import asyncio
import uuid

print("✅ Imports successful.")

✅ Imports successful.


# The four primitives

Most tutorials throw a dozen concepts at you in the first module. ADK only has four that matter on day one.

| Primitive | What it is | What you do with it |
|---|---|---|
| `LlmAgent` | An LLM wired to instructions, a model, and (optionally) tools | You define one |
| `Runner` | The event loop that drives a conversation | You call `.run_async()` |
| `Event` | Every message, tool call, tool response, and state change | You read them to see what's happening |
| `Session` | The conversation's memory — event history plus a state dict | You create one per conversation |

Read the table against your notebook-5 loop: the `Runner` is your Steps 1–5 written once and properly, `Session` is your `messages` list with a real home, and every `print()` you sprinkled into the loop becomes a typed `Event`. Everything else — workflow agents, multi-agent hierarchies, callbacks, memory services, evaluation — composes on top of these four. Keep them in mind; they'll come back every module.

# The World's Simplest Agent

Four required arguments: a name, a model, a description, and an instruction. No tools yet — this is just an LLM with a system prompt, wrapped in ADK's plumbing.

Notice what the next cell does *not* do: it makes no API call. `LlmAgent(...)` only builds a Python object that *describes* an agent — name, which model, what instruction. Nothing talks to OpenRouter until we run it. Keep this in mind; it is the reason the next section needs a few more pieces.


In [6]:
greeter = LlmAgent(
    name="greeter",
    model=LiteLlm(model=MODEL_STRING),
    description="Greets the user in a friendly way.",
    instruction="You are a friendly greeter. Respond in one short sentence.",
)

print(f"✅ Agent '{greeter.name}' built.")

✅ Agent 'greeter' built.


## Running the Agent

An `LlmAgent` on its own does nothing. To have a conversation you need two more pieces — a `Runner` and a `Session` — and a way of running them (`async`) that may look unfamiliar if you've only written plain Python functions so far. Take the questions in the order they come up in your head.

**"Until now I just called functions. Why do I need a Runner?"**
When you called an LLM API directly, you sent one request and got one response — your Python was in charge. An agent is a *loop*: the model answers → maybe it asks for a tool → ADK runs the tool → the result goes back to the model → the model answers again → … until it produces a final answer. Somebody has to run that loop, pass the growing conversation history into every round, and record what happened. That somebody is the `Runner`. You could write the loop yourself (it is ~30 lines), but you would rewrite it for every agent. The Runner is that loop, written once, with all the bookkeeping.

**"What is a Session?"**
The Runner needs to know *which* conversation it is continuing. A session is one conversation: its message history plus a small state dictionary. `InMemorySessionService` keeps sessions in a Python dict — they vanish when the kernel restarts, which is fine for learning. M03 swaps it for a database with one line.

**"What is `async` and why is it needed?"**
Most of an agent's life is spent waiting: for the model's HTTP response, for a tool's API call. `async` is Python's way of saying *"this function will wait on something external; while it waits, let other work run."* ADK is written async-first because a real agent server runs many conversations at once and cannot afford to freeze while one of them waits on the network. For us in a notebook, the practical rules are just two:

1. A function that uses `await` inside must be declared `async def`.
2. You call such a function with `await` in front — `await chat(...)`, not `chat(...)`.

Jupyter and Colab let you write `await` directly in a cell (a plain `.py` script would need `asyncio.run(...)` around it; you'll see that form in M10). That is all you need to drive ADK. You do not have to understand the event loop to use it.

**"What is a stream, and why do I need it?"**
`runner.run_async()` does not return one answer. It returns an *async generator* — something you loop over with `async for` — and it *yields* an `Event` every time something happens: the model produced text, the model asked for a tool, the tool replied. You could wait for the last event only (in production you often do), but seeing every step is how you learn what the agent actually did, and how you debug it when it goes wrong. That is why the helper prints each one.

**"Why is this `chat` function so complex?"**
It isn't, once you see it does four things in order:

1. create a fresh session (so runs don't leak into each other),
2. create a Runner for this agent,
3. wrap your text in a `types.Content` object — ADK's message format: a *role* (`"user"`) plus a list of *parts* (here one text part; later also tool calls and tool results),
4. loop over the event stream and print each event, tagging the last one `[FINAL]`.

Read it once slowly. We reuse it unchanged in every later module, so the investment pays off fourteen times.


In [7]:
APP = "m01_demo"
USER = "student"

session_service = InMemorySessionService()

async def chat(agent, prompt: str):
    """Send one user message to the agent and print every event it emits."""
    # Fresh session per call so previous runs don't leak in.
    sid = f"session-{uuid.uuid4().hex[:8]}"
    await session_service.create_session(app_name=APP, user_id=USER, session_id=sid)
    runner = Runner(agent=agent, app_name=APP, session_service=session_service)

    message = types.Content(role="user", parts=[types.Part(text=prompt)])

    print(f"USER: {prompt}\n")
    async for event in runner.run_async(user_id=USER, session_id=sid, new_message=message):
        tag = "[FINAL]" if event.is_final_response() else "[step]"
        if event.content and event.content.parts:
            for p in event.content.parts:
                if p.text:
                    print(f"{tag} {event.author}: {p.text.strip()}")
                if p.function_call:
                    print(f"[tool_call] {p.function_call.name}({dict(p.function_call.args)})")
                if p.function_response:
                    print(f"[tool_resp] {p.function_response.response}")

await chat(greeter, "Hi, what's your name?")

USER: Hi, what's your name?



[FINAL] greeter: Hi! I’m Greeter—nice to meet you!


One event came back, containing the final text response. That's the minimum a conversation can produce — a user turn goes in, one model turn comes out, the stream ends. **Events are the unit of observability in ADK:** every message, tool call, tool response, state change, or agent hand-off is one. Later modules add tools, delegation, and state mutations; you'll see the event stream grow.

# Add a Tool — And Watch the Events Multiply

A tool in ADK is a plain Python function — and here the mapping table pays out its best row. In the bridge you passed a hand-written 34-line schema dict *plus* the function. Now you pass **just the function**, and delete the schema: ADK builds it from two pieces of Python you already use out of habit:

- **Type hints** — the `: str` after `location` and the `-> dict` after the parentheses. Python itself ignores them at runtime; they are annotations, not checks. ADK reads them to tell the model *"this argument is a string"*. Without them the model has to guess what to pass.
- **Docstring** — the triple-quoted text right under `def`. Normally documentation for humans; here it is sent to the model, nearly verbatim, as the tool's description. The model decides *whether* and *how* to call your tool purely from this text. Write it for the model, not for a code reviewer.

ADK combines the function name, the type hints and the docstring into a **JSON schema** — the standard format in which every LLM provider accepts tool definitions — and sends it along with your message. The model never sees your Python code; it sees only that schema. When it decides a tool is needed, it replies with a structured request ("call `get_weather` with `city='Prague'`"); ADK executes the real function, sends the return value back, and the model writes the final answer.

You'll see three events instead of one:

```
user turn → [tool_call] → [tool_resp] → [FINAL] text response
```


In [8]:
weather_agent = LlmAgent(
    name="weather_agent",
    model=LiteLlm(model=MODEL_STRING),
    description="Reports current weather for a given location.",
    instruction=(
        "You are a weather assistant. When the user asks about weather, "
        "call the get_current_weather tool and report what it returns. Be brief."
    ),
    tools=[get_current_weather],   # the SAME function from the bridge — no schema dict anywhere
)

await chat(weather_agent, "What's the weather in Prague right now? Use Celsius.")


USER: What's the weather in Prague right now? Use Celsius.



[tool_call] get_current_weather({'location': 'Prague', 'format': 'celsius'})
[tool_resp] {'location': 'Prague', 'temperature': 14.0, 'unit': 'celsius', 'condition': 'Cloudy'}


[FINAL] weather_agent: Prague is currently **14°C and cloudy**.


## What Just Happened

The output above is the smallest complete picture of an agent run:

1. **`[tool_call] get_current_weather({'location': 'Prague', 'format': 'celsius'})`** — the model decided a tool was needed and emitted a structured call. ADK caught it before it left the agent.
2. **`[tool_resp] {'location': 'Prague', 'temperature': 14.0, ...}`** — ADK executed the Python function and fed the return value back to the model as a tool-response event.
3. **`[FINAL] weather_agent: ...`** — the model wrote a natural-language answer using the tool's output.

Three events, all visible, all inspectable. If the model had called the tool wrong, you'd see the error. If it had refused to call the tool, you'd see the refusal. If it had called two tools in sequence, you'd see both. This visibility is ADK's single biggest advantage over the hand-rolled loop from the bridge — there your `print()`s were the only window; here every step is a typed event you can filter, log, or test against (M09 does exactly that).

## A Word on `adk web`

So far you have run agents from a notebook, and you will keep doing that in this course because notebooks record cleanly. But ADK also installs a command-line tool, `adk`, and its most useful subcommand is `adk web` — a local developer UI for your agents.

It expects your agent in a folder, as a small Python package:

```
my_agents/
└── weather_agent/
    ├── __init__.py     # one line:  from . import agent
    └── agent.py        # the LlmAgent from above, assigned to a variable named  root_agent
```

Then, from a terminal in the folder that contains `my_agents/` (with your key in a `.env` there, or exported in the shell):

```bash
adk web my_agents
```

It starts a web server on `http://localhost:8000` and opens a chat window. Pick `weather_agent` from the dropdown, type a message, and next to the chat you get the same event stream we just printed — as a clickable timeline with the full JSON of every event: the exact request sent to the model, the tool's arguments and return value, timings. It also shows the session state, which becomes important in M03.

Why we don't use it in the videos: it needs a terminal and a browser tab, which does not fit the notebook format. Why you should use it at home: it is the best debugger you get for free. M10 comes back to the same folder layout, because `adk api_server` and `adk deploy` use it too.


# Your Turn

Four small tasks, five minutes each. Run them in a new cell below so you can diff the event streams.

0. **Your old loop, new vendor.** In the bridge cell, change `OPENAI_MODEL` to
   `"anthropic/claude-haiku-4.5"` and re-run it. Your notebook-5 code just ran on a Claude model
   — that is the OpenRouter trick in one line.
1. **Change the model in ADK.** Replace `MODEL_STRING` with `openrouter/anthropic/claude-haiku-4.5`
   or `openrouter/google/gemini-3.7-flash`. Re-run the weather agent. Does the final answer change?
   Do the events change shape?
2. **Break the tool on purpose.** Ask the weather agent about `"Reykjavik"` (not in `FAKE_WEATHER`).
   Read the events. How does the model handle the error dict — does it pass it through honestly,
   or hallucinate a forecast?
3. **Add your second notebook-5 function.** Bring back `get_n_day_weather_forecast(location, format,
   num_days)` (invent fake data), add it to `tools=[...]`, and ask *"What's the 3-day forecast for
   Munich in Fahrenheit?"*. In notebook 5 you wrote a second 30-line schema for it — count the lines
   you write now.

No grading. The goal is to get a feel for what shows up in the event stream and what doesn't.


# Key Takeaways

- **You had already built an agent** — ADK replaces your hand-written schema (docstring does it), your `available_functions` dict (dispatch), your loop (`Runner`), and your `messages` list (`Session`).
- An ADK agent is a configuration object. Four required arguments — name, model, description, instruction — plus an optional list of tools.
- The **Runner** drives conversations forward; `.run_async()` yields an **Event** per step.
- Every meaningful moment in an agent run is an Event: user input, tool calls, tool responses, state changes, final text. Read events to debug.
- The **`LiteLlm` adapter** is the one-line swap that makes ADK vendor-neutral: ADK's `LiteLlm` class on one side, the independent `litellm` library on the other, OpenRouter as the one-key reseller behind it. The same agent runs on GPT, Claude, Gemini, or any OpenRouter-routable model — the course default is not a Google model on purpose.
- The four primitives to carry forward: **`Agent` · `Runner` · `Event` · `Session`**. Every later module composes on top of them.

# Next up — M02: Tools as verbs

You just saw one flavor of tool: a plain Python function. ADK supports three more — an OpenAPI spec, an MCP server, and another agent wrapped as a tool. M02 builds one of each. Before you move on, make sure you can say out loud what each primitive is. If not, re-read the event-stream output once more.